In [1]:
import pandas as pd
import numpy as np

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

import plotly.graph_objs as go
import plotly.express as px

import utility_functions as uf

In [59]:
path = "data/"
df_country_dist_w1 = pd.read_csv(path+"df_country_dist_w1.csv", index_col=0)
df_country_dist_w1_world = pd.read_csv(path+"df_country_dist_w1_world_norm.csv", index_col=0)
df_country_dist_cosine_world = pd.read_csv(path+"df_country_dist_cosine_world.csv", index_col=0)
df_country_dist_cosine_world_log = pd.read_csv(path+"df_country_dist_cosine_world_log.csv", index_col=0)
df_country_dist_cosine_scaled_world = pd.read_csv(path+"df_country_dist_cosine_scaled_world.csv", index_col=0)

df_country_subfield_norm = pd.read_csv(path+"df_country_subfield_norm.csv", index_col=0)
df_country_subfield_norm_world  = pd.read_csv(path+"df_country_subfield_norm_world.csv", index_col=0)
df_country_subfield_norm_world_log  = pd.read_csv(path+"df_country_subfield_norm_world_log.csv", index_col=0)
df_country_subfield_norm_world_norm = pd.read_csv(path+"df_country_subfield_norm_world_norm.csv", index_col=0)

color_by_domain = {
    "Social Sciences": "red",
    "Health Sciences": "green",
    "Physical Sciences": "blue",
    "Life Sciences": "purple",
}

In [208]:
df_dist = df_country_dist_w1
df_country_subfield = df_country_subfield_norm

In [209]:
# Example: distance matrix (symmetric, zeros on diagonal)
D = df_dist.values

# Convert to condensed form (required by linkage)
condensed_D = squareform(D)

# Perform hierarchical clustering
# method can be: 'single', 'complete', 'average', 'ward'
Z = linkage(condensed_D, method='average')


In [253]:
# Assign clusters by specifying a max distance threshold
labels = fcluster(Z, t=10, criterion='maxclust')
order = np.argsort(labels)
# print("Cluster labels:", labels)

In [254]:
labels = np.array(labels)

order = np.argsort(labels)
D_ordered = D[np.ix_(order, order)]

names = df_dist.index.to_numpy()
names_ordered = [uf.id2name_country[name] for name in names[order]]
labels_ordered = labels[order]

In [255]:
fig = go.Figure(
    data=go.Heatmap(
        z=D_ordered,
        x=names_ordered,
        y=names_ordered,
        colorscale="Viridis",
        colorbar=dict(title="Distance"),
        hovertemplate="Row: %{y}<br>Col: %{x}<br>Dist: %{z:.3f}<extra></extra>"
    )
)

fig.update_layout(
    title="Distance matrix reordered by cluster labels",
    xaxis=dict(
        tickangle=45,
        tickfont=dict(size=8),
        automargin=True
    ),
    yaxis=dict(
        tickfont=dict(size=8),
        automargin=True
    ),
    width=900,
    height=900
)
# cluster boundaries
changes = np.where(np.diff(labels_ordered) != 0)[0] + 1

for c in changes:
    fig.add_shape(
        type="line",
        x0=-0.5, x1=len(names_ordered)-0.5,
        y0=c-0.5, y1=c-0.5,
        line=dict(color="white", width=1)
    )
    fig.add_shape(
        type="line",
        x0=c-0.5, x1=c-0.5,
        y0=-0.5, y1=len(names_ordered)-0.5,
        line=dict(color="white", width=1)
    )

fig.show()


In [256]:
hovertext = [uf.id2subfield_topic[int(x)] for x in df_country_subfield_norm.columns]
hovertext_2d = np.tile(hovertext, (len(names_ordered), 1))

fig = go.Figure(
    data=go.Heatmap(
        z=df_country_subfield.iloc[order, :].values,
        y=names_ordered,
        x=df_country_subfield.columns,
        colorscale="Viridis",
        colorbar=dict(title="Values"),
        zmin=0,
        zmax=0.05,
        text=hovertext_2d,        # 1D list for columns
        hoverinfo='x+y+text+z' # show column name, row, hovertext, and value
    )
)

fig.update_layout(
    title="Norm reordered by cluster labels",
    xaxis=dict(
        tickangle=45,
        tickfont=dict(size=8),
        automargin=True
    ),
    yaxis=dict(
        tickfont=dict(size=8),
        automargin=True
    ),
    width=900,
    height=900
)
# cluster boundaries
changes = np.where(np.diff(labels_ordered) != 0)[0] + 1

for c in changes:
    fig.add_shape(
        type="line",
        y0=c-0.5, y1=c-0.5,
        x0=-0.5, x1=len(df_country_subfield.columns)-0.5,
        line=dict(color="red", width=1)
    )

fig.show()

In [257]:
df_map = (
    df_dist
    .merge(uf.df_country[["alpha-2", "alpha-3", "name", "region", "sub-region"]], left_index=True, right_on="alpha-2", how="left")
    [["alpha-2", "alpha-3", "name", "sub-region", "region"]]
    .assign(cluster=labels,
            cluster_str = lambda df: df["cluster"].astype(str))
    .rename(columns={"alpha-3": "country", "alpha-2": "country2"})
)
df_map.sample(5)

,country2,country,name,sub-region,region,cluster,cluster_str
148.0,ME,MNE,Montenegro,Southern Europe,Europe,3,3
29.0,BW,BWA,Botswana,Sub-Saharan Africa,Africa,2,2
130.0,LU,LUX,Luxembourg,Western Europe,Europe,2,2
183.0,RU,RUS,Russian Federation,Eastern Europe,Europe,6,6
137.0,MT,MLT,Malta,Southern Europe,Europe,3,3


In [258]:
(
    df_map
    [["cluster", "country2"]]
    .groupby("cluster")
    .count()
    .sort_values("country2", ascending=False)
)

,country2
cluster,
3,98
5,46
2,38
6,15
8,8
9,4
10,4
1,3
4,3


In [259]:
country = "IT"
df_map.query(f"country2 == \"{country}\"")

,country2,country,name,sub-region,region,cluster,cluster_str
110.0,IT,ITA,Italy,Southern Europe,Europe,3,3


In [260]:
fig = px.choropleth(
    df_map,
    locations="country",
    color="cluster_str",               # use the categorical version
    locationmode="ISO-3",
    color_discrete_sequence=px.colors.qualitative.Set3,  # discrete color palette
    title="Country clusters based on distance matrix"
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth"
    ),
    height=600,
)

fig.show()

In [273]:
cluster = 3

(
    df_map
    .query(f"cluster == {cluster}")
    .sort_values(["region", "sub-region"])
    .dropna()
)

,country2,country,name,sub-region,region,cluster,cluster_str
3.0,DZ,DZA,Algeria,Northern Africa,Africa,3,3
65.0,EG,EGY,Egypt,Northern Africa,Africa,3,3
127.0,LY,LBY,Libya,Northern Africa,Africa,3,3
226.0,TN,TUN,Tunisia,Northern Africa,Africa,3,3
61.0,DJ,DJI,Djibouti,Sub-Saharan Africa,Africa,3,3
...,...,...,...,...,...,...,...
158.0,NZ,NZL,New Zealand,Australia and New Zealand,Oceania,3,3
74.0,FJ,FJI,Fiji,Melanesia,Oceania,3,3
157.0,NC,NCL,New Caledonia,Melanesia,Oceania,3,3
144.0,FM,FSM,"Micronesia, Federated States of",Micronesia,Oceania,3,3


In [274]:
def count_function(name, df):
    cols = ["top_1", "top_2", "top_3", "top_4", "top_5"]
    return (df[cols] == name).sum().sum()

def top_n_cols(row, n=5):
    return [uf.id2subfield_topic.get(int(code), code) for code in row.nlargest(n).index.tolist()]

def get_cluster_dfs(cluster, df_country_subfield_norm_world_norm, df_map,):
    top_cols_df = df_country_subfield_norm_world_norm.apply(lambda r: top_n_cols(r), axis=1)

    df_top_subfields = pd.DataFrame(
        top_cols_df.tolist(),
        index=df_country_subfield_norm_world_norm.index,
        columns=[f"top_{i+1}" for i in range(5)]
    )

    df_map_top = (
        df_map
        .merge(df_top_subfields, left_on="country2", right_index=True)
        .query(f"cluster == {cluster}")
        .sort_values(["top_1", "top_2", "top_3", "top_4", "top_5"])
    )

    df_unique_subfields = (
        pd.DataFrame(df_map_top[["top_1", "top_2", "top_3", "top_4", "top_5"]].values.flatten(), columns=["subfield"])
        .drop_duplicates()  # keep unique
    )

    df_subfields_counted = (
        df_unique_subfields
        .merge(uf.df_topics[["subfield_name", "field_name"]], left_on="subfield", right_on="subfield_name", how="left")
        .drop(["subfield_name"], axis=1)
        .drop_duplicates(subset=["subfield"])  # ensure one row per subfield
        .assign(count_subfields=lambda df: df.subfield.apply(lambda name: count_function(name, df_map_top)))
    )

    df_fields_counted = (
        df_subfields_counted
        .drop("subfield", axis=1)
        .groupby("field_name")
        .sum()
        .div(len(df_map_top.index) * 5)
        .merge(uf.df_topics[["field_name", "domain_name"]].drop_duplicates(), left_index=True, right_on="field_name")
    )
    return df_top_subfields, df_unique_subfields, df_subfields_counted, df_fields_counted

In [275]:
df_top_subfields, df_unique_subfields, df_subfields_counted, df_fields_counted = get_cluster_dfs(
    cluster,
    df_country_subfield,
    df_map
)

In [276]:
px.bar(df_fields_counted, x="field_name", y="count_subfields", color="domain_name",
       category_orders={"field_name": df_fields_counted.sort_values("count_subfields", ascending=False)["field_name"]},
       color_discrete_map=color_by_domain)